In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

dataset = "mrpc"
seed = 2026
selected_dp_percent = [i for i in range(1, 101)]
iteration = 500


def plot_data_selection(seed, dataset, train_data, g, iteration=500):
    """Plot 9-subplot data-selection figure for one (dataset, train_data) setting.

    Looks up the following variables from g (caller's globals()):
      - seed{seed}_{dataset}_{train_data}_inv_lam1e_6
      - seed{seed}_{dataset}_{train_data}_r{R}_eigen1e_2_inv1e_6     (R in ranks)
      - seed{seed}_{dataset}_{train_data}_r{R}_eigen1e_2_eigen1e_2   (R in ranks)
      - seed{seed}_{dataset}_{train_data}_A_BASE
      - seed{seed}_{dataset}_{train_data}_inv_early_stop
      - seed{seed}_{dataset}_{train_data}_topr_early_stop
      - selected_dp_percent  (shared)
    """
    selected_dp_percent = g["selected_dp_percent"]
    inv             = g[f"seed{seed}_{dataset}_{train_data}_inv_lam1e_6"]
    A_BASE          = g[f"seed{seed}_{dataset}_{train_data}_A_BASE"]
    inv_early_stop  = g[f"seed{seed}_{dataset}_{train_data}_inv_early_stop"]
    topr_early_stop = g[f"seed{seed}_{dataset}_{train_data}_topr_early_stop"]

    ranks = [1, 5, 10, 15, 20, 25, 30]
    eigen_data_1 = [(g[f"seed{seed}_{dataset}_{train_data}_r{r}_eigen1e_2_eigen1e_2"], f"rank {r}%") for r in ranks]
    eigen_data_2 = [(g[f"seed{seed}_{dataset}_{train_data}_r{r}_eigen1e_2_inv1e_6"],   f"rank {r}%") for r in ranks]
    diff_data    = [(np.array(d1) - np.array(d2), lab)
                    for (d1, lab), (d2, _) in zip(eigen_data_1, eigen_data_2)]

    num_dp = train_data
    ZOOM_PCT = 20
    ZOOM_N = ZOOM_PCT

    fig, axes = plt.subplots(3, 3, figsize=(30, 28))
    axes = axes.flatten()

    inv_color = "red"
    eigen_colors = plt.cm.viridis(np.linspace(0.1, 0.9, 7))

    # ===== Subplot (1,1): Eigen mode (shapley + prediction) - 1~ZOOM_PCT% =====
    ax1 = axes[0]
    plot_percent_10 = selected_dp_percent[:ZOOM_N]
    ax1.plot(plot_percent_10, inv[:ZOOM_N], color=inv_color, linewidth=2.5,
             linestyle="-", marker="o", markersize=10, label="INV λ=1e-6", alpha=0.85)
    for i, (data, label) in enumerate(eigen_data_1):
        ax1.plot(plot_percent_10, data[:ZOOM_N], color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="s", markersize=10, label=label, alpha=0.85)

    all_data_10_eigen = [inv[:ZOOM_N]] + [d[0][:ZOOM_N] for d in eigen_data_1]
    y_min_10 = min(min(d) for d in all_data_10_eigen)
    y_max_10 = max(max(d) for d in all_data_10_eigen)
    y_margin_10 = (y_max_10 - y_min_10) * 0.1
    ylim_10_min = y_min_10 - y_margin_10
    ylim_10_max = y_max_10 + y_margin_10

    ax1.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax1.set_ylabel("Accuracy", fontsize=20)
    ax1.set_title(f"Eigen Shapley + Eigen Pred (1-{ZOOM_PCT}%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax1.legend(fontsize=20, loc="best")
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0.5, ZOOM_PCT + 0.5)
    ax1.set_ylim(ylim_10_min, ylim_10_max)
    ax1.tick_params(axis="both", labelsize=18)

    # ===== Subplot (1,2): Eigen shapley + INV prediction - 1~ZOOM_PCT% =====
    ax2 = axes[1]
    ax2.plot(plot_percent_10, inv[:ZOOM_N], color=inv_color, linewidth=2.5,
             linestyle="-", marker="o", markersize=10, label="INV λ=1e-6", alpha=0.85)
    for i, (data, label) in enumerate(eigen_data_2):
        ax2.plot(plot_percent_10, data[:ZOOM_N], color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="^", markersize=10, label=label, alpha=0.85)

    all_data_10_inv = [inv[:ZOOM_N]] + [d[0][:ZOOM_N] for d in eigen_data_2]
    y_min_10_inv = min(min(d) for d in all_data_10_inv)
    y_max_10_inv = max(max(d) for d in all_data_10_inv)
    ylim_10_min = min(ylim_10_min, y_min_10_inv - (y_max_10_inv - y_min_10_inv) * 0.1)
    ylim_10_max = max(ylim_10_max, y_max_10_inv + (y_max_10_inv - y_min_10_inv) * 0.1)

    ax2.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax2.set_ylabel("Accuracy", fontsize=20)
    ax2.set_title(f"Eigen Shapley + INV Pred (1-{ZOOM_PCT}%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax2.legend(fontsize=20, loc="best")
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0.5, ZOOM_PCT + 0.5)
    ax2.set_ylim(ylim_10_min, ylim_10_max)
    ax2.tick_params(axis="both", labelsize=18)

    ax1.set_ylim(ylim_10_min, ylim_10_max)

    # ===== Subplot (1,3): Difference (Eigen - INV) - 1~100% =====
    ax3 = axes[2]
    for i, (data, label) in enumerate(diff_data):
        ax3.plot(selected_dp_percent, data, color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="d", markersize=6, label=label, alpha=0.85)

    all_diff_data = [d[0] for d in diff_data]
    y_min_diff = min(min(d) for d in all_diff_data)
    y_max_diff = max(max(d) for d in all_diff_data)
    y_margin_diff = max(abs(y_min_diff), abs(y_max_diff)) * 0.1
    ylim_diff_min = y_min_diff - y_margin_diff
    ylim_diff_max = y_max_diff + y_margin_diff

    ax3.axhline(y=0, color="black", linestyle="--", linewidth=1.5, alpha=0.5)
    ax3.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax3.set_ylabel("Accuracy Difference", fontsize=20)
    ax3.set_title(f"Difference (Eigen Pred - INV Pred) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax3.legend(fontsize=20, loc="best")
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(0, 101)
    ax3.set_ylim(ylim_diff_min, ylim_diff_max)
    ax3.tick_params(axis="both", labelsize=18)

    # ===== Subplot (2,1): Eigen mode (shapley + prediction) - 1~100% =====
    ax4 = axes[3]
    ax4.plot(selected_dp_percent, inv, color=inv_color, linewidth=2.5,
             linestyle="-", marker="o", markersize=6, label="INV λ=1e-6", alpha=0.85)
    for i, (data, label) in enumerate(eigen_data_1):
        ax4.plot(selected_dp_percent, data, color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="s", markersize=6, label=label, alpha=0.85)

    all_data_100_eigen = [inv] + [d[0] for d in eigen_data_1]
    y_min_100 = min(min(d) for d in all_data_100_eigen)
    y_max_100 = max(max(d) for d in all_data_100_eigen)
    y_margin_100 = (y_max_100 - y_min_100) * 0.1
    ylim_100_min = y_min_100 - y_margin_100
    ylim_100_max = y_max_100 + y_margin_100

    ax4.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax4.set_ylabel("Accuracy", fontsize=20)
    ax4.set_title(f"Eigen Shapley + Eigen Pred (1-100%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax4.legend(fontsize=20, loc="best")
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(0, 101)
    ax4.set_ylim(ylim_100_min, ylim_100_max)
    ax4.tick_params(axis="both", labelsize=18)

    # ===== Subplot (2,2): Eigen shapley + INV prediction - 1~100% =====
    ax5 = axes[4]
    ax5.plot(selected_dp_percent, inv, color=inv_color, linewidth=2.5,
             linestyle="-", marker="o", markersize=6, label="INV λ=1e-6", alpha=0.85)
    for i, (data, label) in enumerate(eigen_data_2):
        ax5.plot(selected_dp_percent, data, color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="^", markersize=6, label=label, alpha=0.85)

    all_data_100_inv = [inv] + [d[0] for d in eigen_data_2]
    y_min_100_inv = min(min(d) for d in all_data_100_inv)
    y_max_100_inv = max(max(d) for d in all_data_100_inv)
    ylim_100_min = min(ylim_100_min, y_min_100_inv - (y_max_100_inv - y_min_100_inv) * 0.1)
    ylim_100_max = max(ylim_100_max, y_max_100_inv + (y_max_100_inv - y_min_100_inv) * 0.1)

    ax5.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax5.set_ylabel("Accuracy", fontsize=20)
    ax5.set_title(f"Eigen Shapley + INV Pred (1-100%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax5.legend(fontsize=20, loc="best")
    ax5.grid(True, alpha=0.3)
    ax5.set_xlim(0, 101)
    ax5.set_ylim(ylim_100_min, ylim_100_max)
    ax5.tick_params(axis="both", labelsize=18)

    ax4.set_ylim(ylim_100_min, ylim_100_max)

    # ===== Subplot (2,3): Early Stop Ratio =====
    ax6 = axes[5]
    method_names = ["INV"] + [f"rank {r}%" for r in ranks]
    method_values = [inv_early_stop[0]] + list(topr_early_stop)
    bar_colors = [inv_color] + list(eigen_colors)
    x_pos = np.arange(len(method_names))
    bars = ax6.bar(x_pos, method_values, color=bar_colors, alpha=0.85, edgecolor="black", linewidth=1.2)
    for bar, value in zip(bars, method_values):
        ax6.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                 f"{value:.2f}", ha="center", va="bottom", fontsize=20, fontweight="bold")
    ax6.set_xlabel("Method", fontsize=20)
    ax6.set_ylabel("Early Stop Ratio (%)", fontsize=20)
    ax6.set_title(f"Early Stop Ratio (%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax6.set_xticks(x_pos)
    ax6.set_xticklabels(method_names, fontsize=20)
    ax6.grid(True, alpha=0.3, axis="y")
    ax6.tick_params(axis="y", labelsize=18)

    # ===== Subplot (3,1): ER for Eigen Pred - 1~ZOOM_PCT% =====
    ax7 = axes[6]
    inv_zoom_arr = np.array(inv[:ZOOM_N])

    er_data_eigen = []
    for i, (data, label) in enumerate(eigen_data_1):
        ers = (np.array(data[:ZOOM_N]) - A_BASE) / (inv_zoom_arr - A_BASE)
        er_data_eigen.append((ers, f"{label} ({np.mean(ers):.3f}±{np.std(ers):.3f})"))

    for i, (ers, label) in enumerate(er_data_eigen):
        ax7.plot(plot_percent_10, ers, color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="s", markersize=10, label=label, alpha=0.85)
    ax7.axhline(y=1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.5, label="y=1 (baseline)")
    ax7.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax7.set_ylabel("Excess Retention (ER)", fontsize=20)
    ax7.set_title(f"ER: Eigen Pred (1-{ZOOM_PCT}%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax7.legend(fontsize=20, loc="best")
    ax7.grid(True, alpha=0.3)
    ax7.set_xlim(0.5, ZOOM_PCT + 0.5)
    ax7.tick_params(axis="both", labelsize=18)

    # ===== Subplot (3,2): ER for INV Pred - 1~ZOOM_PCT% =====
    ax8 = axes[7]
    er_data_inv = []
    for i, (data, label) in enumerate(eigen_data_2):
        ers = (np.array(data[:ZOOM_N]) - A_BASE) / (inv_zoom_arr - A_BASE)
        er_data_inv.append((ers, f"{label} ({np.mean(ers):.3f}±{np.std(ers):.3f})"))

    for i, (ers, label) in enumerate(er_data_inv):
        ax8.plot(plot_percent_10, ers, color=eigen_colors[i], linewidth=2.5,
                 linestyle="-", marker="^", markersize=10, label=label, alpha=0.85)
    ax8.axhline(y=1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.5, label="y=1 (baseline)")
    ax8.set_xlabel("Selected Data Percentage (%)", fontsize=20)
    ax8.set_ylabel("Excess Retention (ER)", fontsize=20)
    ax8.set_title(f"ER: INV Pred (1-{ZOOM_PCT}%) | Data: {num_dp}, Iter: {iteration}", fontsize=20)
    ax8.legend(fontsize=20, loc="best")
    ax8.grid(True, alpha=0.3)
    ax8.set_xlim(0.5, ZOOM_PCT + 0.5)
    ax8.tick_params(axis="both", labelsize=18)

    # ===== Subplot (3,3): Turn off =====
    axes[8].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:

# ===== Configuration =====
num_dp = 1000
# =========================

seed2026_mrpc_1000_A_BASE = 

seed2026_mrpc_1000_inv_lam1e_6           = 
# Prediction with top shapley values (shapley eigen 1e-2/prediction inv 1e-6)
seed2026_mrpc_1000_r1_eigen1e_2_inv1e_6  = 
seed2026_mrpc_1000_r5_eigen1e_2_inv1e_6  = 
seed2026_mrpc_1000_r10_eigen1e_2_inv1e_6 = 
seed2026_mrpc_1000_r15_eigen1e_2_inv1e_6 = 
seed2026_mrpc_1000_r20_eigen1e_2_inv1e_6 = 
seed2026_mrpc_1000_r25_eigen1e_2_inv1e_6 = 
seed2026_mrpc_1000_r30_eigen1e_2_inv1e_6 = 

# Prediction with top shapley values (shapley eigen 1e-2/prediction eigen 1e-2)
seed2026_mrpc_1000_r1_eigen1e_2_eigen1e_2  = 
seed2026_mrpc_1000_r5_eigen1e_2_eigen1e_2  = 
seed2026_mrpc_1000_r10_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_1000_r15_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_1000_r20_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_1000_r25_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_1000_r30_eigen1e_2_eigen1e_2 = 

seed2026_mrpc_1000_inv_early_stop  = []
seed2026_mrpc_1000_topr_early_stop = []

plot_data_selection(seed, dataset, num_dp, globals(), iteration=iteration)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ===== Configuration =====
num_dp = 2000
# =========================

seed2026_mrpc_2000_A_BASE = 

seed2026_mrpc_2000_inv_lam1e_6           = 
# Prediction with top shapley values (shapley eigen 1e-2/prediction inv 1e-6)
seed2026_mrpc_2000_r1_eigen1e_2_inv1e_6  = 
seed2026_mrpc_2000_r5_eigen1e_2_inv1e_6  = 
seed2026_mrpc_2000_r10_eigen1e_2_inv1e_6 = 
seed2026_mrpc_2000_r15_eigen1e_2_inv1e_6 = 
seed2026_mrpc_2000_r20_eigen1e_2_inv1e_6 = 
seed2026_mrpc_2000_r25_eigen1e_2_inv1e_6 = 
seed2026_mrpc_2000_r30_eigen1e_2_inv1e_6 = 

# Prediction with top shapley values (shapley eigen 1e-2/prediction eigen 1e-2)
seed2026_mrpc_2000_r1_eigen1e_2_eigen1e_2  = 
seed2026_mrpc_2000_r5_eigen1e_2_eigen1e_2  = 
seed2026_mrpc_2000_r10_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_2000_r15_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_2000_r20_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_2000_r25_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_2000_r30_eigen1e_2_eigen1e_2 = 

seed2026_mrpc_2000_inv_early_stop  = []
seed2026_mrpc_2000_topr_early_stop = []

plot_data_selection(seed, dataset, num_dp, globals(), iteration=iteration)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ===== Configuration =====
num_dp = 3668
# =========================

seed2026_mrpc_3668_A_BASE = 3162

seed2026_mrpc_3668_inv_lam1e_6           = [6814, 6936, 7157, 7181, 7475, 7328, 7402, 7549, 7525, 7500, 7451, 7500, 7525, 7549, 7549, 7598, 7696, 7868, 7868, 7843, 7819, 7917, 7917, 8039, 7917, 7966, 7892, 7990, 8064, 8064, 8137, 7990, 8064, 8064, 8113, 8088, 8088, 8162, 8088, 8088, 8064, 8137, 8260, 8309, 8358, 8456, 8505, 8529, 8554, 8529, 8554, 8529, 8554, 8554, 8529, 8603, 8578, 8578, 8603, 8554, 8554, 8578, 8578, 8529, 8554, 8382, 8456, 8554, 8627, 8529, 8676, 8701, 8603, 8603, 8603, 8456, 8505, 8480, 8333, 8382, 8431, 8358, 8284, 8260, 8235, 8235, 8235, 8260, 8260, 8113, 8137, 8015, 7941, 7819, 7696, 7549, 7623, 7206, 7010, 6471]
# Prediction with top shapley values (shapley eigen 1e-2/prediction inv 1e-6)
seed2026_mrpc_3668_r1_eigen1e_2_inv1e_6  = [6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6863, 6863, 6838, 6985, 7010, 7034, 7034, 7034, 7059, 7059, 7059, 7059, 7108, 7157, 7108, 7132, 7132, 7108, 7108, 7132, 7181, 7181, 7132, 7083, 7108, 7108, 7083, 7108, 7108, 7108, 7157, 7181, 7181, 7157, 7157, 7230, 7206, 7255, 7157, 7181, 7157, 7157, 7255, 7157, 7255, 7083, 7157, 7181, 7181, 7181, 7230, 7255, 7181, 7108, 7279, 7181, 7353, 7132, 7279, 7181, 7230, 7255, 7181, 7181, 7083, 7010, 6887, 7059, 7059, 6936, 6985, 6863, 6912, 6961, 6838, 6642, 6618, 6765, 6765, 6863, 6838, 6961, 6667, 6667, 6471]
seed2026_mrpc_3668_r5_eigen1e_2_inv1e_6  = [6838, 6863, 6863, 6863, 6838, 6863, 6961, 6936, 6863, 6863, 7010, 6936, 7059, 7083, 7034, 7083, 7034, 7108, 7181, 7230, 7230, 7255, 7157, 7132, 7181, 7157, 7206, 7255, 7279, 7328, 7377, 7353, 7377, 7279, 7328, 7304, 7377, 7426, 7426, 7451, 7353, 7328, 7402, 7451, 7475, 7475, 7500, 7475, 7574, 7574, 7574, 7451, 7549, 7623, 7598, 7623, 7696, 7696, 7574, 7574, 7647, 7647, 7647, 7647, 7745, 7721, 7672, 7819, 7745, 7721, 7623, 7598, 7574, 7696, 7475, 7328, 7353, 7402, 7377, 7402, 7426, 7157, 7181, 7108, 7279, 7230, 6961, 6912, 6985, 7083, 7034, 7083, 6936, 6912, 6936, 6569, 6740, 6642, 6520, 6471]
seed2026_mrpc_3668_r10_eigen1e_2_inv1e_6 = 
seed2026_mrpc_3668_r15_eigen1e_2_inv1e_6 = [6936, 6936, 6961, 6961, 6961, 7034, 7034, 6985, 6985, 7010, 7083, 7083, 7059, 7010, 7034, 7083, 7108, 7157, 7132, 7255, 7206, 7304, 7402, 7353, 7402, 7353, 7475, 7574, 7549, 7574, 7426, 7525, 7598, 7525, 7549, 7623, 7770, 7745, 7843, 7843, 7819, 7794, 7770, 7819, 7745, 7770, 7917, 7868, 7843, 7843, 7819, 7819, 7843, 7794, 7721, 7696, 7868, 7941, 7917, 7917, 7941, 7990, 7990, 7843, 7819, 7892, 7868, 7745, 7819, 7941, 7941, 7892, 7941, 8015, 7794, 7892, 7745, 7721, 7770, 7892, 7672, 7721, 7721, 7721, 7721, 7328, 7500, 7475, 7328, 7279, 7328, 7010, 7010, 6912, 6838, 6814, 6740, 6716, 6520, 6471]
seed2026_mrpc_3668_r20_eigen1e_2_inv1e_6 = 
seed2026_mrpc_3668_r25_eigen1e_2_inv1e_6 = 
seed2026_mrpc_3668_r30_eigen1e_2_inv1e_6 = [7108, 7034, 7108, 7083, 7059, 6985, 7083, 7108, 7083, 7059, 7034, 7083, 7132, 7181, 7108, 7132, 7157, 7132, 7279, 7353, 7377, 7402, 7402, 7549, 7574, 7721, 7696, 7745, 7745, 7696, 7794, 7843, 7868, 7941, 7892, 7892, 7819, 7819, 7819, 7843, 7794, 7794, 7868, 7819, 7794, 7721, 7892, 7843, 7892, 7966, 7966, 7868, 7819, 7941, 7990, 7892, 7966, 8015, 8015, 8162, 8113, 8284, 8309, 8260, 8284, 8309, 8309, 8235, 8284, 8186, 8162, 8186, 8235, 8211, 8186, 8235, 8162, 8162, 8186, 8088, 8015, 8064, 7966, 7917, 7794, 7819, 7819, 7672, 7451, 7353, 7426, 7402, 7377, 7279, 7034, 6936, 6887, 6838, 6667, 6471]

# Predction with top shapley values (shapley eigen 1e-2/prediction eigen 1e-2)
seed2026_mrpc_3668_r1_eigen1e_2_eigen1e_2  = [6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6838, 6912, 6912, 6961, 6936, 6961, 6912, 6912, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7034, 7034, 7034, 7034, 7034, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7010, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7034, 7083, 7083, 7083, 7132, 7132, 7132, 7132, 7132, 7132, 7132, 7157, 7206, 7181, 7181, 7181, 7157, 7157, 7181, 7157, 7181, 7181, 7083, 7108, 7157, 7132, 7108]
seed2026_mrpc_3668_r5_eigen1e_2_eigen1e_2  = [6838, 6863, 6838, 6936, 6912, 6863, 6912, 6936, 6912, 6912, 6985, 6985, 7034, 6985, 7010, 6985, 7034, 7059, 7010, 7157, 7132, 7108, 7108, 7132, 7132, 7108, 7083, 7181, 7206, 7206, 7230, 7206, 7206, 7230, 7230, 7230, 7230, 7230, 7230, 7230, 7230, 7157, 7230, 7206, 7181, 7206, 7206, 7206, 7181, 7181, 7181, 7206, 7206, 7181, 7206, 7206, 7206, 7206, 7181, 7181, 7157, 7157, 7157, 7157, 7181, 7206, 7181, 7181, 7181, 7230, 7230, 7206, 7255, 7255, 7255, 7279, 7279, 7279, 7328, 7304, 7328, 7353, 7328, 7353, 7402, 7475, 7451, 7426, 7451, 7451, 7451, 7525, 7574, 7525, 7451, 7402, 7328, 7353, 7328, 7230]
seed2026_mrpc_3668_r10_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_3668_r15_eigen1e_2_eigen1e_2 = [7010, 6961, 7034, 6961, 6961, 7083, 7083, 7108, 7108, 7132, 7206, 7181, 7206, 7279, 7304, 7304, 7304, 7328, 7377, 7304, 7255, 7328, 7377, 7328, 7353, 7328, 7328, 7353, 7304, 7377, 7304, 7328, 7377, 7377, 7402, 7353, 7451, 7475, 7475, 7525, 7500, 7525, 7525, 7549, 7574, 7672, 7623, 7574, 7574, 7598, 7598, 7549, 7549, 7549, 7525, 7475, 7549, 7574, 7598, 7549, 7574, 7623, 7647, 7623, 7647, 7721, 7696, 7696, 7672, 7696, 7696, 7672, 7770, 7794, 7843, 7794, 7917, 7966, 7966, 7941, 7966, 7966, 7941, 7917, 7941, 8015, 8064, 7966, 8015, 8015, 7990, 7966, 7941, 7892, 7819, 7721, 7574, 7525, 7426, 7108]
seed2026_mrpc_3668_r20_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_3668_r25_eigen1e_2_eigen1e_2 = 
seed2026_mrpc_3668_r30_eigen1e_2_eigen1e_2 = [7108, 7034, 7108, 7083, 7083, 6985, 7059, 7083, 7059, 7034, 7059, 7157, 7157, 7206, 7108, 7108, 7157, 7181, 7328, 7353, 7598, 7574, 7598, 7525, 7623, 7672, 7770, 7696, 7745, 7794, 7868, 7770, 7770, 7745, 7819, 7843, 7917, 7892, 7892, 7819, 7794, 7819, 7868, 7868, 7843, 7843, 7892, 7917, 8015, 8088, 8039, 8039, 8015, 7990, 7966, 8064, 8015, 8039, 8015, 8015, 8088, 8113, 8113, 8137, 8137, 8137, 8162, 8186, 8186, 8162, 8211, 8186, 8162, 8162, 8186, 8235, 8260, 8382, 8309, 8284, 8235, 8211, 8162, 8137, 8186, 8211, 8211, 8162, 8088, 8039, 8015, 7941, 7966, 7868, 7917, 7892, 7819, 7721, 7574, 7230]

seed2026_mrpc_3668_inv_early_stop  = [90.37]
seed2026_mrpc_3668_topr_early_stop = [27.01, 54.28,      , 63.07,      ,      , 78.51]

plot_data_selection(seed, dataset, num_dp, globals(), iteration=iteration)


SyntaxError: invalid syntax (684456075.py, line 15)